In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver;

CREATE OR REPLACE TEMP VIEW customers_typed AS
SELECT
  customer_id,
  customer_unique_id as customer_unique_id,
  customer_zip_code_prefix as customer_zip_code_prefix,
  customer_state,
  customer_city,
  _ingested_at
FROM workspace.bronze.customers
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY _ingested_at DESC) = 1;

CREATE OR REPLACE TABLE workspace.silver.customers AS
SELECT * FROM customers_typed
WHERE customer_id IS NOT NULL AND customer_unique_id IS NOT NULL AND LENGTH(customer_state) = 2 AND customer_zip_code_prefix  RLIKE '^[0-9]{5}$';

CREATE OR REPLACE TABLE workspace.silver.customers_quarantine AS
SELECT *
, CASE
    WHEN customer_id IS NULL OR customer_unique_id IS NULL
      THEN 'llave nula'
    WHEN customer_state IS NULL OR LENGTH(customer_state) <> 2
      THEN 'estado no valido'
    WHEN customer_zip_code_prefix IS NULL OR NOT customer_zip_code_prefix  RLIKE '^[0-9]{5}$' 
      THEN 'codigo zip invalido'
  END AS motivo
FROM customers_typed
WHERE 1=1 AND (
  customer_id IS NULL 
  OR customer_unique_id IS NULL 
  OR customer_zip_code_prefix IS NULL 
  OR NOT customer_zip_code_prefix  RLIKE '^[0-9]{5}$'
  OR customer_state IS NULL OR LENGTH(customer_state) <> 2);

In [0]:
%sql
Select 'bronze' as layer, count(*) from workspace.bronze.customers
UNION ALL
Select 'silver' as layer, count(*) from workspace.silver.customers
UNION all
Select 'quarantine' as layer, count(*) from workspace.silver.customers_quarantine